##Silver Transformation of the circuits bronze table
### 1. Read the table from bronze schema

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run ../00.Common/02.Helper_Notebook

In [0]:
source_name = f"{catalog_name}.{bronze_schema}.circuits"
target_name = f"{catalog_name}.{silver_schema}.circuits"

In [0]:
circuits_df = spark.table(source_name)
display(circuits_df)

### 2. Keep only the required columns for analysis 

In [0]:
from pyspark.sql import functions as F
circuits_required_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitname"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("SourceFile")
    )

### 3. Rename the columns

In [0]:
circuits_renamed_df = circuits_required_df.withColumnsRenamed({"circuitId":"circuits_id","circuitname":"circuit_name","lat":"latitude","long":"longitude","SourceFile":"source_file"})
display(circuits_renamed_df)

### 4. Removing duplicates and NULL from the dataset

In [0]:
# Removing the null values using sql and column expressions
# circuits_clean_df = circuits_renamed_df.filter(
#     "circuit_id IS NOT NULL"
# )
circuits_clean_df = circuits_renamed_df.filter(F.col("circuits_id").isNotNull())

In [0]:
# Removing duplicates based on the primary key
circuits_clean_df1 = circuits_clean_df.dropDuplicates(["circuits_id"])
display(circuits_clean_df1)

### 5. Transforming the column values 

In [0]:
# Converting the values to initcap format in locality and circuit name columns
circuits_final_df = (circuits_clean_df1
                     .withColumn("circuit_name",F.initcap(F.col("circuit_name")))
                     .withColumn("locality",F.initcap(F.col("locality")))
)
display(circuits_final_df)

### 6. Writing the final dataframe as table into the silver schema

In [0]:
(
    circuits_final_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(target_name)
)

In [0]:
%sql
select * from formula1.silver.circuits;